In [1]:
# 本程序采用机器学习中 监督学习、深度学习、及 混合无监督学习 的主流算法对单一指标进行日频择时策略
# 所有模型分为两类：分类Classification/回归Regression
# 分类模型预测下一交易日是否买入（-1或1），以及预测正确的准确率；回归模型直接预测下一交易日的收益为多少（一个确定的值）
# 分类算法采用accuracy和AUC作为评判指标，回归算法采用R2_score和RMSE作为评判指标
# 训练集为前2000天，测试集为后900+天（目前没有验证集及开发集）


'''
程序的基本结构：
Step1: 数据Preprocessing
Step2: 监督学习模型, 测试了LR, Logistic Classification, SVR/SVM, Random Forest Regressor / Classifier, Adaboost Algo
Step3: 深度学习模型, 测试了Perceptron以 及 MLP)
Step4: 深度学习模型 时许特征模型--RNN / LSTM （仍在尝试）
Step5: 无监督学习进行自主分类，然后利用监督/深度学习表现较好的方法进行 Regression / Classification
Step6: 可视化, 包含AUC对比图、收益比率对比图、累计收益对比图
'''


# Import all libraries
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)


import math
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.ticker as mtick
import matplotlib.cm as cm
from pca import pca
from scipy.stats import zscore
import scipy.cluster.hierarchy as sch
from scipy.cluster import vq 
from mpl_toolkits.axes_grid1 import ImageGrid
from scipy.stats import multivariate_normal as normal 
from sklearn import cluster, tree, svm, datasets
from sklearn.cluster import DBSCAN, MiniBatchKMeans, KMeans
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC, LinearSVC, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Perceptron, LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.manifold import TSNE, MDS
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import silhouette_samples, silhouette_score, roc_curve, auc, roc_auc_score, mean_squared_error
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score, recall_score, r2_score
from sklearn.multiclass import OneVsRestClassifier
import random
import pdb
import time
from tqdm import tqdm 
from itertools import cycle
from typing import Tuple

random.seed(20230721)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity='all'
from IPython import display
# %matplotlib inline

# Data Preprocessing
df = pd.read_excel('database/中证国防指数技术指标.xlsx')
df = df.iloc[:, :-3].drop(columns = 'TICKER')
y = df.iloc[:, 5].shift(1)
y.iloc[0] = 0
# y = y.iloc[:-1]
X = df.drop(columns=['DT', 'ICHANGE'])
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.3, random_state = 233)
# global X_train, X_val, y_train, y_val
X_train = X.iloc[:2000, :]
X_val = X.iloc[2000:, :]
y_train = y.iloc[:2000]
y_val = y.iloc[2000:]
standard_scalor = StandardScaler()
X_train = standard_scalor.fit_transform(X_train)
X_val = standard_scalor.fit_transform(X_val)

classifier_models = ['Logistic', 'SVM', 'Tree', 'Forest', 'Adaboost', 'Perceptron', 'MLP', "Log_cluster", "MLP_cluster"]
regressor_models = ["LR", "SVR", "Forest", "Adaboost", "MLP", "LR_cluster"]

global classifier_lst, regressor_lst
# accuracy_classifier = []
# auc_classifier = []
# y_test_sum_classifier = []
# y_test_prob_sum_classifier = []
classifier_lst = [[], [], [], []]

# rmse_regressor = []
# auc_regressor = []
# y_test_sum_regressor = []
# y_test_weight_sum_regressor = []
regressor_lst = [[], [], [], []]

y_max = 0
for i in range(len(y_val)):
    if pd.DataFrame(y_val).iloc[i, 0] > 0:
        y_max += pd.DataFrame(y_val).iloc[i, 0]
print("所有日期的累积收益（基准参考）:", y_val.sum() + y_train.sum())
print("测试集的基准收益:", y_val.sum())
print("测试集可获得的最大收益:", y_max)

# 所有日期的累积收益（所有ICHANGE的和）: 543.4685999999996
# 测试集的基准收益（测试集内所有ICHANGE的和）: 302.55330000000015
# 测试集可获得的最大收益（测试集所有ICHANGE>0的值的和）: 13657.346099999986

所有日期的累积收益（基准参考）: 543.4685999999996
测试集的基准收益: 302.55330000000015
测试集可获得的最大收益: 13657.346099999986


In [5]:
# Step 2
def Linear_Regressor(*args, **kwargs):
    y_test = y_val
    X_test = X_val
    linear_regression = LinearRegression()
    linear_regression.fit(X_train, y_train)
    y_pred = linear_regression.predict(X_test)
    if len(args) != 0:
        return y_pred
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    r_squared = r2_score(y_test, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_val.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)

    positive_y_pred_count = np.sum(y_pred > 0)
    positive_y_val_values = y_val[y_pred > 0]
    positive_y_val_count = np.sum(positive_y_val_values > 0)
    probability = positive_y_val_count / positive_y_pred_count if positive_y_pred_count != 0 else 0

    print("多重线性回归(作为检验之后最理想的模型)下日频交易胜率约为:", round(probability, 4))
    
    return regressor_lst

def Logistic_Classfication(*args, **kwargs):
    X_test = X_val
    lr = LogisticRegression(solver = 'liblinear')
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    lr.fit(X_train, y_train_bin)
    y_pred = lr.predict(X_val)
    if len(args) != 0:
        return y_pred
    acc = accuracy_score(y_test_bin, y_pred)
    # print("The accuracy under Logistic Regression model is:", acc)
    y_preb_probs = lr.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, pd.DataFrame(y_preb_probs).iloc[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    value_for_index_2 = (y_pred == 1) * y_val.squeeze()
    classifier_lst[2].append(value_for_index_2.sum() / y_max)
    # value_for_index_3 = (y_pred == 1) * y_val.squeeze() * y_preb_probs[:, 1]
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(multiplier)
    return classifier_lst

def SVM():
    X_test = X_val
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    svm = SVC(probability=True)
    svm.fit(X_train, y_train_bin.values.ravel())  # Use ravel to convert column vector to 1D array
    y_pred = svm.predict(X_test)

    acc = accuracy_score(y_test_bin, y_pred)
    y_pred_probs = svm.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    # print('Linear SVM validation accuracy = {:0.1f}%'.format(100*acc))
    # print('Linear SVM auc =', auc)
    
    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def SVR_regressor():
    X_test = X_val
    y_test = y_val
    svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.2)
    svr.fit(X_train, y_train)
    y_pred = svr.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    r_squared = r2_score(y_test, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_val.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)
    return regressor_lst

def single_decision_tree_classifier():
    X_test = X_val
    y_train_bin = np.where(y_train >= 0, 1, -1)
    y_test_bin = np.where(y_val >= 0, 1, -1)
    clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=8)
    clf = clf.fit(X_train, y_train_bin)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test_bin, y_pred)
    y_pred_probs = clf.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def random_forest_classifier():
    X_test = X_val
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    clf2 = RandomForestClassifier(criterion='gini', max_depth=20, min_samples_leaf=8)
    clf2 = clf2.fit(X_train, y_train_bin)
    y_pred = clf2.predict(X_test)
    y_pred_probs = clf2.predict_proba(X_test)

    acc = accuracy_score(y_test_bin, y_pred)
    y_pred_probs = clf2.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def random_forest_regressor():
    X_test = X_val
    y_test = y_val
    clf2 = RandomForestRegressor(criterion='squared_error', max_depth=20, min_samples_leaf=8)
    clf2 = clf2.fit(X_train, y_train)
    y_pred = clf2.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    r_squared = r2_score(y_test, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_val.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)
    return regressor_lst

def adaboost_classifier():
    X_test = X_val
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    bdt = AdaBoostClassifier(
        tree.DecisionTreeClassifier(max_depth = 12), algorithm = "SAMME", n_estimators = 100, learning_rate = 0.5
    )
    bdt.fit(X_train, y_train_bin)
    y_pred = bdt.predict(X_test)
    acc = accuracy_score(y_test_bin, y_pred)
    y_pred_probs = bdt.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def adaboost_regressor():
    X_test = X_val
    y_test = y_val
    # Create and fit the AdaBoost regressor
    # Note: we are using DecisionTreeRegressor instead of DecisionTreeClassifier
    bdt = AdaBoostRegressor(
        DecisionTreeRegressor(max_depth=12), 
        n_estimators=100, 
        learning_rate=0.5
    )
    bdt.fit(X_train, y_train) 
    y_pred = bdt.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    r_squared = r2_score(y_test, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_val.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)
    return regressor_lst

def perceptron():
    X_test = X_val
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    clf3 = Perceptron(tol = 1e-3, random_state = 0)
    clf3.fit(X_train, y_train_bin.values.ravel())
    y_pred = clf3.predict(X_test)
    acc = accuracy_score(y_test_bin, y_pred)
    auc = roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr")

    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def MLP_classifier(*args, **kwargs):
    X_test = X_val
    y_train_bin = pd.DataFrame(np.where(y_train >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_val >= 0, 1, -1))
    mlp = MLPClassifier(hidden_layer_sizes = (12, 12, 12 ), max_iter = 500, activation='identity')
    mlp.fit(X_train, y_train_bin)
    mlp.score(X_test, y_test_bin)
    y_pred = mlp.predict(X_test)
    if len(args) != 0:
        return y_pred
    y_pred_probs = mlp.predict_proba(X_test)
    acc = accuracy_score(y_test_bin, y_pred)
    y_pred_probs = mlp.predict_proba(X_test)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    yield_ratio = (y_pred == 1) * y_val.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_val)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_val.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)
    return classifier_lst

def MLP_regressor():
    X_test = X_val
    y_test = y_val
    mlp_regressor = MLPRegressor(hidden_layer_sizes=(12, 12, 12), max_iter=500, activation='identity')
    mlp_regressor.fit(X_train, y_train)
    y_pred = mlp_regressor.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    r_squared = r2_score(y_test, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_val.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_val)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_val.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)
    return regressor_lst

# define 2 dimensional reduction method: PCA & t-SNE
def PCA_plot():
    pca1 = PCA(whiten = True)
    pca1.fit(X_train)
    var = pca1.explained_variance_ratio_
    plt.bar(range(1, len(var) + 1), var)
    plt.xlabel('Principal Components')
    plt.ylabel('Explained variance (in decimal)')
    plt.title('Explained variance by each component (PC1, PC2, ...)')
    plt.savefig('database/PCA_components_1_and_2.png')
    # plt.show()
    plt.clf()

    model = pca(n_components = 2)
    results = model.fit_transform(X_train)
    fig, ax = model.scatter()
    plt.savefig('database/PCA_by_components.png')
    # plt.show()
    plt.close()

def tsne_plot(perplex, X_data, y_data):
    # 经测试，perplexity >=30 且 <=100 的时候降维效果相对较好
    print("Plotting TSNE figure under perplexity = {} ...".format(perplex))
    tsne = TSNE(n_components=2, perplexity=perplex)
    X_tsne = tsne.fit_transform(X_data)
    
    # 根据y_train的值为点指定颜色
    colors = np.where(y_data > 0, 'b', 'r')
    tsne_dataframe = pd.DataFrame(data=X_tsne, columns=['tsne comp. 1', 'tsne comp. 2'])
    
    plt.scatter(tsne_dataframe.iloc[:, 0], tsne_dataframe.iloc[:, 1], c=colors, cmap="brg", s=2)
    plt.title('t-SNE with perplexity = ' + str(perplex) + ' has KL divergence: ' + str(round(tsne.kl_divergence_, 5)))
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.savefig('database/TSNE_perplexity_{}.png'.format(perplex))
    # plt.show()
    plt.clf()

# define 2 methods for clustering (kmeans clustering): Silhouette and elbow curve
def Silhouette_kmeans_PCA():
    # 降维后聚类效果：Silhouette (under PCA)
    print("Plotting Silhouette Analysis under PCA...")
    pca = PCA(n_components = 3)
    pca_std = pca.fit_transform(X_train)
    pca_std_df = pd.DataFrame(data = pca_std, columns = ['PC1', 'PC2','PC3'])
    X_pca = pca_std_df.to_numpy()

    range_n_clusters = [2, 3, 4, 5, 6]

    for n_clusters in range_n_clusters:
        # Create a subplot with 1 row and 2 columns
        fig, (ax1, ax2) = plt.subplots(1, 2)
        fig.set_size_inches(18, 7)

        # The 1st subplot is the silhouette plot
        # The silhouette coefficient can range from -1, 1 but in this example all
        # lie within [-0.1, 1]
        ax1.set_xlim([-0.1, 1])
        # The (n_clusters+1)*10 is for inserting blank space between silhouette
        # plots of individual clusters, to demarcate them clearly.
        ax1.set_ylim([0, len(X_pca) + (n_clusters + 1) * 10])

        # Initialize the clusterer with n_clusters value and a random generator
        # seed of 10 for reproducibility.
        clusterer = KMeans(n_clusters = n_clusters, random_state = 10)
        cluster_labels = clusterer.fit_predict(X_pca)

        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        silhouette_avg = silhouette_score(X_pca, cluster_labels)
        # print(
        #     "For n_clusters =",
        #     n_clusters,
        #     "The average silhouette_score is :",
        #     silhouette_avg,
        # )
        '''
        print(
            "For n_clusters =",
            n_clusters,
            "The total sum of distance(squared) is :",
            clusterer.inertia_,
        )
        '''
        # Compute the silhouette scores for each sample
        sample_silhouette_values = silhouette_samples(X_pca, cluster_labels)

        y_lower = 10
        for i in range(n_clusters):
            # Aggregate the silhouette scores for samples belonging to
            # cluster i, and sort them
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

            ith_cluster_silhouette_values.sort()

            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_silhouette_values,
                facecolor = color,
                edgecolor = color,
                alpha = 0.7,
            )

            # Label the silhouette plots with their cluster numbers at the middle
            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            # Compute the new y_lower for next plot
            y_lower = y_upper + 10  # 10 for the 0 samples

        ax1.set_title("The silhouette plot for the various clusters.")
        ax1.set_xlabel("The silhouette coefficient values")
        ax1.set_ylabel("Cluster label")

        # The vertical line for average silhouette score of all the values
        ax1.axvline(x = silhouette_avg, color = "red", linestyle = "--")

        ax1.set_yticks([])  # Clear the yaxis labels / ticks
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        # 2nd Plot showing the actual clusters formed
        colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
        ax2.scatter(
            X_pca[:, 0], X_pca[:, 1], marker = ".", s = 30, lw = 0, alpha = 0.7, c = colors, edgecolor = "k"
        )

        # Labeling the clusters
        centers = clusterer.cluster_centers_
        # Draw white circles at cluster centers
        ax2.scatter(
            centers[:, 0],
            centers[:, 1],
            marker = "o",
            c = "white",
            alpha = 1,
            s = 200,
            edgecolor = "k",
        )

        for i, c in enumerate(centers):
            ax2.scatter(c[0], c[1], marker = "$%d$" % i, alpha = 1, s = 50, edgecolor = "k")

        ax2.set_title("The visualization of the clustered data.")
        ax2.set_xlabel("Feature space for the 1st feature")
        ax2.set_ylabel("Feature space for the 2nd feature")

        plt.suptitle(
            "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d with silhouette_score = %.5f" % (n_clusters, silhouette_avg),
            fontsize=14,
            fontweight="bold",
        )

        plt.savefig('database/Clustering_Sihouette_PCA_{}_clusters.png'.format(n_clusters))
        plt.clf();
        # plt.show();

def Silhouette_kmeans_tsne():
    # 降维后聚类效果：Silhouette (under t-SNE)
    print("Plotting Silhouette Analysis under TSNE...")
    tsne = TSNE(n_components=2, perplexity = 20)
    X_tsne = tsne.fit_transform(X_train)
    tsne_std_df = pd.DataFrame(data = X_tsne, columns = ['tsne comp. 1', 'tsne comp. 2'])
    X_tsne = tsne_std_df.to_numpy()

    range_n_clusters = [2, 3, 4, 5, 6]

    for n_clusters in range_n_clusters:
        # Create a subplot with 1 row and 2 columns
        fig, (ax1, ax2) = plt.subplots(1, 2)
        fig.set_size_inches(18, 7)

        # The 1st subplot is the silhouette plot
        # The silhouette coefficient can range from -1, 1 but in this example all
        # lie within [-0.1, 1]
        ax1.set_xlim([-0.1, 1])
        # The (n_clusters+1)*10 is for inserting blank space between silhouette
        # plots of individual clusters, to demarcate them clearly.
        ax1.set_ylim([0, len(X_tsne) + (n_clusters + 1) * 10])

        # Initialize the clusterer with n_clusters value and a random generator
        # seed of 10 for reproducibility.
        clusterer = KMeans(n_clusters = n_clusters, random_state = 10)
        cluster_labels = clusterer.fit_predict(X_tsne)

        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        silhouette_avg = silhouette_score(X_tsne, cluster_labels)
        # print(
        #     "For n_clusters =",
        #     n_clusters,
        #     "The average silhouette_score is :",
        #     silhouette_avg,
        # )
        '''
        print(
            "For n_clusters =",
            n_clusters,
            "The total sum of distance(squared) is :",
            clusterer.inertia_,
        )
        '''
        # Compute the silhouette scores for each sample
        sample_silhouette_values = silhouette_samples(X_tsne, cluster_labels)

        y_lower = 10
        for i in range(n_clusters):
            # Aggregate the silhouette scores for samples belonging to
            # cluster i, and sort them
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

            ith_cluster_silhouette_values.sort()

            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_silhouette_values,
                facecolor = color,
                edgecolor = color,
                alpha = 0.7,
            )

            # Label the silhouette plots with their cluster numbers at the middle
            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            # Compute the new y_lower for next plot
            y_lower = y_upper + 10  # 10 for the 0 samples

        ax1.set_title("The silhouette plot for the various clusters.")
        ax1.set_xlabel("The silhouette coefficient values")
        ax1.set_ylabel("Cluster label")

        # The vertical line for average silhouette score of all the values
        ax1.axvline(x = silhouette_avg, color = "red", linestyle = "--")

        ax1.set_yticks([])  # Clear the yaxis labels / ticks
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        # 2nd Plot showing the actual clusters formed
        colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
        ax2.scatter(
            X_tsne[:, 0], X_tsne[:, 1], marker = ".", s = 30, lw = 0, alpha = 0.7, c = colors, edgecolor = "k"
        )

        # Labeling the clusters
        centers = clusterer.cluster_centers_
        # Draw white circles at cluster centers
        ax2.scatter(
            centers[:, 0],
            centers[:, 1],
            marker = "o",
            c = "white",
            alpha = 1,
            s = 200,
            edgecolor = "k",
        )

        for i, c in enumerate(centers):
            ax2.scatter(c[0], c[1], marker = "$%d$" % i, alpha = 1, s = 50, edgecolor = "k")

        ax2.set_title("The visualization of the clustered data.")
        ax2.set_xlabel("Feature space for the 1st feature")
        ax2.set_ylabel("Feature space for the 2nd feature")

        plt.suptitle(
            "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d with silhouette_score = %.5f" % (n_clusters, silhouette_avg),
            fontsize = 14,
            fontweight = "bold",
        )
        plt.savefig('database/Clustering_Sihouette_tsne_{}_clusters.png'.format(n_clusters))
        # plt.show();
        plt.clf()

def elbow_curve():
    # Plot the Elbow curve to decide the optimal k value
    print("Plotting Elbow curve...")
    distortions = np.zeros(len(X_train))
    for k in range(1, 10):
        kmeans = cluster.KMeans(k)
        kmeans.fit(X_train)
        distortions[k-1] = kmeans.inertia_
    # Plot the results.
    plt.plot(np.arange(1, 11, 1), distortions[: 10], 'b-x', lw = 2)
    plt.xlabel(r'Clusters $k$')
    plt.ylabel(r'Distortion')
    plt.title(r'Elbow Curve for K-means')
    plt.grid()
    # plt.show();
    plt.clf()
    # The slope of Elbow curve somehow verifies that the optimal k value could be 4 

# supervised models after clustering; clustering 采用 TSNE + kmeans 的方法
def data_clustering():
    tss = TSNE(n_components = 2, method = 'barnes_hut')  # use method = 'exact' if n_component >= 4; component为维度 并不为cluster数
    tss_std = tss.fit_transform(X_train)
    tss_std_df = pd.DataFrame(data=tss_std, columns=['tsne comp. 1', 'tsne comp. 2'])
    X_tss=tss_std_df

    def get_clusters(X_train: pd.DataFrame, X_test: pd.DataFrame, n_clusters: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        applies k-means clustering to training data to find clusters and predicts them for the test set
        """
        clustering = KMeans(n_clusters=n_clusters, random_state = 20230721)
        cluster_labels=clustering.fit_predict(X_tss)
        #train_labels = clustering.labels_
        X_train_clstrs = X_train.copy()
        X_train_clstrs['clusters'] = cluster_labels

        # predict labels on the test set
        test_labels = clustering.fit_predict(X_test)
        X_test_clstrs = X_test.copy()
        X_test_clstrs['clusters'] = test_labels
        return X_train_clstrs, X_test_clstrs
    X_train_clstrs, X_test_clstrs = get_clusters(pd.DataFrame(X_train), pd.DataFrame(X_val), 4)

    X_train_unsup = X_train_clstrs.to_numpy().astype('float')
    X_test_unsup = X_test_clstrs.to_numpy().astype('float')
    y_train_unsup = y_train.astype('float')
    y_test_unsup = y_val.astype('float')

    return X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup

def Linear_Regressor_after_clustering(*args, **kwargs):
    X_train_unsup = args[0]
    X_test_unsup = args[1]
    y_train_unsup = args[2]
    y_test_unsup = args[3]
    linear_regression = LinearRegression()
    linear_regression.fit(X_train_unsup, y_train_unsup)
    y_pred = linear_regression.predict(X_test_unsup)
    if len(args) == 4:
        return y_pred
    rmse = np.sqrt(np.mean((y_test_unsup - y_pred)**2))
    r_squared = r2_score(y_test_unsup, y_pred)
    regressor_lst[0].append(r_squared)
    regressor_lst[1].append(rmse / 100)
    value_for_index_2 = (y_pred >= 0) * y_test_unsup.squeeze()
    regressor_lst[2].append(value_for_index_2.sum() / y_max)
    multiplier = 1
    for i in range(len(y_test_unsup)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_test_unsup.to_list()[i]) / 1000)
    regressor_lst[3].append(multiplier)
    
    return regressor_lst

def Logistic_Classfication_after_clustering(*args, **kwargs):
    X_train_unsup = args[0]
    X_test_unsup = args[1]
    y_train_unsup = args[2]
    y_test_unsup = args[3]
    lr = LogisticRegression(solver = 'liblinear')
    y_train_bin = pd.DataFrame(np.where(y_train_unsup >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_test_unsup >= 0, 1, -1))
    lr.fit(X_train_unsup, y_train_bin)
    y_pred = lr.predict(X_test_unsup)
    if len(args) == 4:
        return y_pred
    acc = accuracy_score(y_test_bin, y_pred)
    # print("The accuracy under Logistic Regression model is:", acc)
    y_preb_probs = lr.predict_proba(X_test_unsup)
    auc = max(roc_auc_score(y_test_bin, pd.DataFrame(y_preb_probs).iloc[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    value_for_index_2 = (y_pred == 1) * y_test_unsup.squeeze()
    classifier_lst[2].append(value_for_index_2.sum() / y_max)
    # value_for_index_3 = (y_pred == 1) * y_val.squeeze() * y_preb_probs[:, 1]
    multiplier = 1
    for i in range(len(y_test_unsup)):
        if y_pred[i] > 0:
            multiplier *= ((1000 + y_test_unsup.to_list()[i]) / 1000)
    classifier_lst[3].append(multiplier)
    return classifier_lst

def MLP_classifier_after_clustering(*args, **kwargs):
    X_train_unsup = args[0]
    X_test_unsup = args[1]
    y_train_unsup = args[2]
    y_test_unsup = args[3]
    y_train_bin = pd.DataFrame(np.where(y_train_unsup >= 0, 1, -1))
    y_test_bin = pd.DataFrame(np.where(y_test_unsup >= 0, 1, -1))
    mlp = MLPClassifier(hidden_layer_sizes = (12, 12, 12 ), max_iter = 500, activation='identity')
    mlp.fit(X_train_unsup, y_train_bin)
    # mlp.score(X_test_unsup, y_test_bin)
    y_pred = mlp.predict(X_test_unsup)
    if len(args) == 4:
        return y_pred
    y_pred_probs = mlp.predict_proba(X_test_unsup)
    acc = accuracy_score(y_test_bin, y_pred)
    auc = max(roc_auc_score(y_test_bin, y_pred_probs[:, 1], average="weighted", multi_class="ovr"), roc_auc_score(y_test_bin, y_pred, average="weighted", multi_class="ovr"))

    yield_ratio = (y_pred == 1) * y_test_unsup.squeeze()
    classifier_lst[0].append(acc)
    classifier_lst[1].append(auc)
    classifier_lst[2].append(yield_ratio.sum() / y_max)

    cumulative_return = 1
    for i in range(len(y_test_unsup)):
        if y_pred[i] == 1:
            cumulative_return *= ((1000 + y_test_unsup.to_list()[i]) / 1000)
    classifier_lst[3].append(cumulative_return)

    return classifier_lst

# Auxillary Ploting function
def classification_result_plt(classifier_lst, classifier_models):
    # models = ['Logistic', 'SVM', 'Decision Tree', 'Random Forest', 'adaboost', 'perceptron', 'MLP']
    print("Plotting final result under classification models...")
    accuracy = classifier_lst[0]
    auc = classifier_lst[1]
    y_test_sum = classifier_lst[2]
    y_test_prob_sum = classifier_lst[3]

    x = np.arange(len(classifier_models))
    width = 0.12
    fig, ax = plt.subplots(figsize = (12, 8))

    rects1 = ax.bar(x - width*1.5, accuracy, width, label='Accuracy') # 交易胜率
    rects2 = ax.bar(x - width/2, auc, width, label='AUC') # 模型“可信度”
    rects3 = ax.bar(x + width/2, y_test_sum, width, label='Return %') # 预测获得收益/可获得最大收益
    rects4 = ax.bar(x + width*1.5, [i / 1000 for i in y_test_prob_sum], width, label='Return coef (per 1000)') # 累乘收益倍数

    ax.set_ylabel('Scores')
    ax.set_title('Scores by model and metric')
    ax.set_xticks(x)
    ax.set_xticklabels(classifier_models)
    ax.legend()
    fig.tight_layout()
    plt.savefig("database/classification_result对比图.png")
    # plt.show()
    plt.clf()

def regression_result_plt(regressor_lst, regressor_models):
    print("Plotting final result under regression models...")
    accuracy = regressor_lst[0]
    auc = regressor_lst[1]
    y_test_sum = regressor_lst[2]
    y_test_prob_sum = regressor_lst[3]

    x = np.arange(len(regressor_models))
    width = 0.18
    fig, ax = plt.subplots(figsize = (12, 8))

    rects1 = ax.bar(x - width*1.5, accuracy, width, label='Accuracy') # 交易胜率
    rects2 = ax.bar(x - width/2, auc, width, label='RMSE (per 100)') # 模型“可信度”; RMSE / 100
    rects3 = ax.bar(x + width/2, y_test_sum, width, label='Return %') # 预测获得收益/可获得最大收益
    rects4 = ax.bar(x + width*1.5, [i / 10000 for i in y_test_prob_sum], width, label='Return coef (per 10000)') # 累乘收益倍数

    ax.set_ylabel('Scores')
    ax.set_title('Scores by model and metric')
    ax.set_xticks(x)
    ax.set_xticklabels(regressor_models)
    ax.legend()
    fig.tight_layout()
    plt.savefig("database/regression_result对比图.png")
    # plt.show()
    plt.clf()

# define plt function to visualize the return along time
def compute_series(y_val, y_pred):
    y_val = np.array(y_val)
    cumulative_positive_y_val = np.zeros_like(y_val)
    cumulative_y_val = np.zeros_like(y_val)
    cumulative_predicted_y_val = np.zeros_like(y_val)
    for i in range(len(y_val)):
        cumulative_y_val[i] = cumulative_y_val[i-1] + y_val[i] if i > 0 else y_val[i]
        
        cumulative_positive_y_val[i] = (cumulative_positive_y_val[i-1] + y_val[i] if y_val[i] > 0 else cumulative_positive_y_val[i-1]) if i > 0 else max(0, y_val[i])
        
        cumulative_predicted_y_val[i] = (cumulative_predicted_y_val[i-1] + y_val[i] if y_pred[i] >= 0 else cumulative_predicted_y_val[i-1]) if i > 0 else max(0, y_val[i] if y_pred[i] >= 0 else 0)
    
    return cumulative_y_val, cumulative_positive_y_val, cumulative_predicted_y_val

def return_plt(model_name, *args):
    if len(args) > 0:
        X_train_unsup = args[0]
        X_test_unsup = args[1]
        y_train_unsup = args[2]
        y_test_unsup = args[3]
    if model_name == "Logistic Classifier":
        y_pred = Logistic_Classfication(model_name)
    elif model_name == "MLP Classifier":
        y_pred = MLP_classifier(model_name)
    elif model_name == "MLR":
        y_pred = Linear_Regressor(model_name)
    elif model_name == "Log_cluster":
        y_pred = Logistic_Classfication_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)
    elif model_name == "MLP_cluster":
        y_pred = MLP_classifier_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)
    elif model_name == "LR_cluster":
        y_pred = Linear_Regressor_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)
    else:
        print("Warning! Invalid model!")
    cumulative_y_val, cumulative_positive_y_val, cumulative_predicted_y_val = compute_series(y_val, y_pred)
    
    fig, ax1 = plt.subplots(figsize=(12,8))
    
    ax1.plot(cumulative_y_val, label='Cumulative y_val', color='green')
    ax1.plot(cumulative_positive_y_val, label='Cumulative positive y_val', color='blue')
    ax1.plot(cumulative_predicted_y_val, label='Cumulative predicted y_val', color='red')
    
    ax2 = ax1.twinx()
    
    # red for y_pred = 1 and blue for y_pred = -1
    colors = np.where(y_pred >= 0, 'red', 'blue')
    
    ax2.bar(range(len(y_val)), y_val, color=colors, alpha=0.4)
    
    ax1.set_xlabel('Days')
    ax1.set_ylabel('Cumulative Returns', color='black')
    ax2.set_ylabel('Daily Returns', color='black')
    
    ax1.grid()
    ax1.legend(loc='upper left')
    ax2.legend(['Daily Returns'], loc='upper right')
    
    plt.savefig("database/{}_收益及时点图.png".format(model_name))
    # plt.show()
    plt.clf()

In [ ]:
def main():
    print("Running ML models ...")
    regressor_lst = Linear_Regressor()
    classifier_lst = Logistic_Classfication()
    classifier_lst = SVM()
    regressor_lst = SVR_regressor()
    classifier_lst = single_decision_tree_classifier()
    classifier_lst = random_forest_classifier()
    regressor_lst = random_forest_regressor()
    classifier_lst = adaboost_classifier()
    regressor_lst = adaboost_regressor()
    classifier_lst = perceptron()
    classifier_lst = MLP_classifier()
    regressor_lst = MLP_regressor()

    # 选取了分类Classification模型中表现较好的Logistic 和 MLP，以及回归Regression模型中表现显著的Multi-Linear Regression (MLR)
    # 图中有三条线和每天的一个bar
    # 上线表示正y_val的逐日累积（如果当天的y_val为正 则加上之前的累积值，如果为负数则累计值不变）; 累积最大收益
    # 下线表示每天的y_val的累积（无论y_val为正或负，用当天的y_val加上之前的累计值）；基准收益
    # 中根线表示预测收益的累积（如果y_pred为1，则用当天对应的y_val加上之前的累积值）；预测模型累积收益
    # bar的含义：如果当天y_pred为1，则用红色的bar表示y_val当天的值；如果当天的y_pred为-1，则用蓝色的bar表示y_val当天的值
    # 换言之，bar为红色代表买入时点（可以解释为开盘买入，收盘卖出），bar为蓝色则当日不进行买卖
    print("Plotting return figures for models ...")
    return_plt("Logistic Classifier")
    return_plt("MLP Classifier")
    return_plt("MLR")

    # Running unsupervised-learning steps ... 
    PCA_plot()
    for i in [10, 20, 30, 50, 100, 200]:
        tsne_plot(i, X_train, y_train)
    Silhouette_kmeans_PCA()
    Silhouette_kmeans_tsne()
    elbow_curve()

    X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup = data_clustering()
    regressor_lst = Linear_Regressor_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup, "verbose")
    classifier_lst = Logistic_Classfication_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup, "verbose")
    classifier_lst = MLP_classifier_after_clustering(X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup, "verbose")
    print("Plotting return figures for models (after clustering)...")
    return_plt("Log_cluster", X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)
    return_plt("MLP_cluster", X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)
    return_plt("LR_cluster", X_train_unsup, X_test_unsup, y_train_unsup, y_test_unsup)

    classification_result_plt(classifier_lst, classifier_models)
    regression_result_plt(regressor_lst, regressor_models)

    print("All finished!")
    
if __name__ == "__main__":
    main()


Running ML models ...
多重线性回归(作为检验之后最理想的模型)下日频交易胜率约为: 0.5191
[[-4.410296615609875e+22, -4.410296615609875e+22, -4.410296615609875e+22, -4.410296615609875e+22, -4.410296615609875e+22, -4.410296615609875e+22], [81850348598.95142, 81850348598.95142, 81850348598.95142, 81850348598.95142, 81850348598.95142, 81850348598.95142], [0.09464655801612888, 0.09464655801612888, 0.09464655801612888, 0.09464655801612888, 0.09464655801612888, 0.09464655801612888], [2.40671021001583, 2.40671021001583, 2.40671021001583, 2.40671021001583, 2.40671021001583, 2.40671021001583]]


In [ ]:
'''
总结：
# 尽管各模型的accuracy并不高，但错误预测多数来源于收益绝对值很小的数据点，而绝对值较大的数据点更容易被准确预测，因此累计收益较为可观
# 无监督学习对于有明确目标（日频收益）的数据有一定的负面影响
# 多重线性回归的结果最好；综合“LR_cluster_收益及时点图.png”，此方法下胜率高达75%+，且对于绝对值大的收益预测显著准确（接近于0的收益的判别错误率较高）

改进&发现：
# 训练组和测试组的分组并不完全合理（目前是前2000为训练组，后900+组为测试组），股票在不同社会环境和热度下会有不一样的反应机制，同时在时间变化下 投资的观念也可能发生很大的变化
# 混合无监督学习对后续分类/预测的优化作用并不大，但我们可以发现数据点在t-SNE算法下perplexity=20至30有一定的的clustering聚类效果。初步猜想与之前研报中提到的市场状态有关（之前的研报把市场状态分为了四类）
# 从结果来看，深度学习的效果并不比简单的逻辑回归和多重线性回归普遍更好，因此推测各指标之间的多层相关性并不复杂
# RNN和LSTM还在测试中！
'''